In [1]:
import ee
import geemap
import matplotlib.pyplot as plt
from datetime import datetime

#Initialize Earth Engine (authenticate if needed)
ee.Authenticate(auth_mode='localhost')  # Uncomment if you haven't authenticated yet
ee.Initialize(project='ee-hamimzakyh')

# Define Sydney coordinates and region of interest
sydney_coords = [151.2093, -33.8688]  # [longitude, latitude]
roi = ee.Geometry.Point(sydney_coords).buffer(50000)  # 50km buffer around Sydney

# Define the date range for June 2025
start_date = '2025-06-01'
end_date = '2025-06-30'

# Filter Landsat 8/9 collection
landsat = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2') \
    .filterBounds(roi) \
    .filterDate(start_date, end_date) \
    .sort('CLOUD_COVER') \
    .first()

# Check if data exists
if landsat is None:
    print("No Landsat data found for this date range. Trying Landsat 8...")
    landsat = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
        .filterBounds(roi) \
        .filterDate(start_date, end_date) \
        .sort('CLOUD_COVER') \
        .first()

if landsat is None:
    print("No Landsat data available for June 2025 in Sydney region")
else:
    # Get image info
    image_date = ee.Date(landsat.get('DATE_ACQUIRED')).format('YYYY-MM-dd').getInfo()
    cloud_cover = landsat.get('CLOUD_COVER').getInfo()
    print(f"Image Date: {image_date}, Cloud Cover: {cloud_cover}%")
    
    # Create visualization with True Color (RGB)
    # Landsat 8/9 bands: B2=Blue, B3=Green, B4=Red
    vis_params = {
        'bands': ['SR_B4', 'SR_B3', 'SR_B2'],
        'min': 7000,
        'max': 20000
    }
    
   # Display using geemap
Map = geemap.Map(center=sydney_coords, zoom=10)
Map.addLayer(landsat, vis_params, f'Landsat {image_date}')
Map.addLayer(roi, {}, 'Region of Interest')
Map  # This displays the map automatically in Jupyter

ModuleNotFoundError: No module named 'ee'